# Training from Scratch Guide

**Note**: This notebook is designed to work with pre-trained checkpoints. If you don't have checkpoints, you'll need to train the models first.

## Training Requirements

The GDSS (Graph Diffusion via Score-based generative modeling using Structural information) model requires:

1. **Training Data**: QM9, ZINC250k, or custom graph datasets
2. **Training Config**: A config file with training parameters (learning rate, epochs, etc.)
3. **Training Script**: A script to train the score-based diffusion model

## Training Steps

### 1. Create a Training Configuration

Create a file `config/train_qm9.yaml` with training parameters:

```yaml
data:
  data: QM9
  dir: './data'
  batch_size: 1024
  max_node_num: 9
  max_feat_num: 4
  init: atom

train:
  name: gdss_qm9_training
  num_epochs: 3000
  lr: 0.001
  lr_schedule: True
  lr_decay: 0.999
  ema: 0.999
  weight_decay: 0.0
  reduce_mean: False
  eps: 1.0e-5
  grad_clip: 1.0

sde:
  x:
    type: VPSDE
    beta_min: 0.1
    beta_max: 20.0
    num_scales: 1000
  adj:
    type: VPSDE
    beta_min: 0.1
    beta_max: 20.0
    num_scales: 1000

model:
  x:
    type: ScoreNetworkX
    depth: 3
    nhid: 128
    num_linears: 3
    c_init: 2
    c_hid: 8
    c_final: 4
    adim: 32
    num_heads: 4
    conv: 'GCN'
  adj:
    type: ScoreNetworkA
    depth: 3
    nhid: 128
    num_linears: 3
    c_init: 2
    c_hid: 8
    c_final: 4
    adim: 32
    num_heads: 4
    conv: 'GCN'

seed: 42
```

### 2. Create a Training Script

Since this repository doesn't include a training script, you would need to create one. Here's a minimal training script structure:

```python
# train.py
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.gdss.parsers.config import get_config
from src.gdss.utils.loader import load_device, load_data, load_sde, load_model_optimizer
from src.gdss.core.losses import get_sde_loss_fn
from src.gdss.utils.ema import ExponentialMovingAverage

def train(config):
    # Load device
    device = load_device()
    
    # Load data
    train_loader, test_loader = load_data(config, get_graph_list=False)
    
    # Load SDEs
    sde_x = load_sde(config.sde.x)
    sde_adj = load_sde(config.sde.adj)
    
    # Load models
    model_x, optimizer_x = load_model_optimizer(config.model.x, config.train, device)
    model_adj, optimizer_adj = load_model_optimizer(config.model.adj, config.train, device)
    
    # Setup EMA
    ema_x = ExponentialMovingAverage(model_x.parameters(), decay=config.train.ema)
    ema_adj = ExponentialMovingAverage(model_adj.parameters(), decay=config.train.ema)
    
    # Loss function
    loss_fn = get_sde_loss_fn(
        sde_x, sde_adj,
        train=True,
        reduce_mean=config.train.reduce_mean,
        eps=config.train.eps
    )
    
    # Training loop
    for epoch in range(config.train.num_epochs):
        model_x.train()
        model_adj.train()
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.train.num_epochs}')
        for x, adj in pbar:
            x, adj = x.to(device), adj.to(device)
            
            # Forward pass
            loss_x, loss_adj = loss_fn(model_x, model_adj, x, adj)
            loss = loss_x + loss_adj
            
            # Backward pass
            optimizer_x.zero_grad()
            optimizer_adj.zero_grad()
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model_x.parameters(), config.train.grad_clip)
            torch.nn.utils.clip_grad_norm_(model_adj.parameters(), config.train.grad_clip)
            
            optimizer_x.step()
            optimizer_adj.step()
            
            # Update EMA
            ema_x.update()
            ema_adj.update()
            
            pbar.set_postfix({'loss': loss.item()})
        
        # Save checkpoint
        if (epoch + 1) % 100 == 0:
            torch.save({
                'epoch': epoch,
                'model_x_state_dict': model_x.state_dict(),
                'model_adj_state_dict': model_adj.state_dict(),
                'optimizer_x_state_dict': optimizer_x.state_dict(),
                'optimizer_adj_state_dict': optimizer_adj.state_dict(),
                'ema_x': ema_x.state_dict(),
                'ema_adj': ema_adj.state_dict(),
                'config': config
            }, f'checkpoints/QM9/gdss_qm9_epoch_{epoch+1}.pth')

if __name__ == '__main__':
    config = get_config('train_qm9', seed=42)
    train(config)
```

### 3. Run Training

```bash
python train.py
```

### 4. Download Pre-trained Checkpoints (Alternative)

If you don't want to train from scratch, you can:

1. Check the original GDSS paper repository: https://github.com/harryjo97/GDSS
2. Look for pre-trained checkpoints in their releases or supplementary materials
3. Contact the authors for pre-trained weights

### 5. Update Notebook to Use Your Checkpoint

Once trained, update cell 4 in this notebook:

```python
config_file = 'sample_qm9'  # Uses the checkpoint specified in config
```

Make sure your `config/sample_qm9.yaml` has:
```yaml
ckpt: gdss_qm9_epoch_3000  # or whatever your checkpoint is named
```

## Important Notes

- **Training Time**: Training diffusion models can take 24-72 hours on a good GPU
- **Data**: You'll need the QM9 dataset downloaded and preprocessed
- **Resources**: Requires significant GPU memory (8GB+ recommended)
- **Checkpoints**: Will be saved in `checkpoints/QM9/` directory

## Quick Start Without Training

For this notebook to work immediately, you need to obtain pre-trained checkpoints from the GDSS authors or train your own following the steps above.

In [13]:
import os
import sys
import torch
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pickle
from easydict import EasyDict as edict

# Add src directory to path for imports
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from gdss.core.losses import get_score_fn
from gdss.core.solver_guidance import Predictor, LangevinCorrector, NoneCorrector
from gdss.utils.graph_utils import mask_adjs, mask_x, gen_noise
from tqdm.notebook import trange

from gdss.parsers.config import get_config

from gdss.utils.logger import Logger, set_log, start_log, train_log, sample_log, check_log
from gdss.utils.loader import load_ckpt, load_data, load_seed, load_device, load_model_from_ckpt, \
                         load_ema_from_ckpt, load_sde, load_yaml_config
from gdss.utils.graph_utils import adjs_to_graphs, init_flags, quantize_mol, \
                              compute_group_assignments, count_across_community_edges, est_p_intra_inter, \
                              is_sbm_graph
from gdss.utils.mol_utils import gen_mol, mols_to_smiles, load_smiles, canonicalize_smiles, mols_to_nx

# Note: moses and evaluation modules need to be installed separately
try:
    from moses.metrics.metrics import get_all_metrics
except ImportError:
    print("Warning: moses package not installed. Install with: pip install moses")

try:
    from evaluation.stats import eval_graph_list
except ImportError:
    print("Warning: evaluation module not found")

from gdss.core.losses_guidance import compute_dp1, compute_dp2, compute_nodedp1, compute_nodedp2

# Data and preamble

In [14]:
device = load_device()
device = [0]

In [16]:
config_file = 'sample_qm9' # sample_sbm sample_community_small
seed = 0
config = get_config(config_file, seed)

In [18]:
# -------- Load checkpoint --------
ckpt_dict = load_ckpt(config, device)
configt = ckpt_dict['config']

FileNotFoundError: [Errno 2] No such file or directory: './checkpoints/QM9/gdss_qm9.pth'

In [17]:
load_seed(configt.seed)
train_graph_list, _ = load_data(configt, get_graph_list=True)
with open(f'data/{configt.data.data.lower()}_test_nx.pkl', 'rb') as f:
    test_graph_list = pickle.load(f)                                   # for NSPDK MMD

NameError: name 'configt' is not defined

In [ ]:
train_smiles, test_smiles = load_smiles(configt.data.data)
train_smiles, test_smiles = canonicalize_smiles(train_smiles), canonicalize_smiles(test_smiles)

In [ ]:
log_folder_name, log_dir, _ = set_log(configt, is_train=False)
log_name = f"{config.ckpt}-sample-guidance"
logger = Logger(str(os.path.join(log_dir, f'{log_name}.log')), mode='a')

In [ ]:
f, axs = plt.subplots(5,5, figsize=(10, 10))
for ax in axs.ravel():
    nx.draw(test_graph_list[0], with_labels=False, ax=ax, node_size=10)

In [ ]:
if not check_log(log_folder_name, log_name):
    logger.log(f'{log_name}')
    start_log(logger, configt)
    train_log(logger, configt)
sample_log(logger, config)

In [ ]:
# -------- Load models --------
model_x = load_model_from_ckpt(ckpt_dict['params_x'], ckpt_dict['x_state_dict'], device)
model_adj = load_model_from_ckpt(ckpt_dict['params_adj'], ckpt_dict['adj_state_dict'], device)

In [ ]:
if config.sample.use_ema:
    ema_x = load_ema_from_ckpt(model_x, ckpt_dict['ema_x'], configt.train.ema)
    ema_adj = load_ema_from_ckpt(model_adj, ckpt_dict['ema_adj'], configt.train.ema)
    
    ema_x.copy_to(model_x.parameters())
    ema_adj.copy_to(model_adj.parameters())

In [ ]:
print(f'GEN SEED: {config.sample.seed}')
load_seed(config.sample.seed)

# Generation

In [ ]:
sde_x = load_sde(configt.sde.x)
sde_adj = load_sde(configt.sde.adj)
max_node_num  = configt.data.max_node_num

device_id = f'cuda:{device[0]}' if isinstance(device, list) else device
print(device_id)

In [ ]:
continuous = True

predictor = config.sampler.predictor
corrector = config.sampler.corrector

snr=config.sampler.snr
scale_eps=config.sampler.scale_eps
n_steps=config.sampler.n_steps

probability_flow = config.sample.probability_flow
eps = config.sample.eps
denoise = config.sample.noise_removal

In [ ]:
if configt.data.data in ['QM9', 'ZINC250k']: # CHANGED
    batch_size = 10000
    # batch_size = configt.data.batch_size
else:
    batch_size = configt.data.batch_size
shape_x = (batch_size, max_node_num, configt.data.max_feat_num)
shape_adj = (batch_size, max_node_num, max_node_num)

In [ ]:
init_flags_iter = init_flags(train_graph_list, configt, batch_size=batch_size).to(device_id)

In [ ]:
init_x = sde_x.prior_sampling(shape_x).to(device_id)
init_adj = sde_adj.prior_sampling_sym(shape_adj).to(device_id)

In [ ]:
class ReverseDiffusionPredictor(Predictor):
    def __init__(self, obj, sde, score_fn, probability_flow=False, guidance_args=None):
        super().__init__(sde, score_fn, probability_flow)
        self.obj = obj

        self.guidance_args = guidance_args

        self.Z = None

    def guidance(self, x, adj, flags, t, is_adj):
        obj = adj if is_adj else x

        dt = -1. / self.rsde.N
        timestep = (t * (self.rsde.N - 1) / self.rsde.T).long()

        loss_fn = eval(self.guidance_args.loss_fn)
        loss_kwargs = self.guidance_args.get('loss_kwargs', {})

        if self.obj == 'adj' and self.Z is None:
            # Assign half and half of the nodes randomly to each group
            n_elems = adj.shape[1] // 2
            # Create a template row with the correct number of elements
            template_row = np.array([0] * n_elems + [1] * (adj.shape[1] - n_elems))

            # Create an array where each row is a copy of the template row
            idxs_com = np.tile(template_row, (adj.shape[0], 1))

            # Apply a random permutation along the columns for each row
            for i in range(adj.shape[0]):
                np.random.shuffle(idxs_com[i])
            Zs = torch.nn.functional.one_hot(torch.tensor(idxs_com), num_classes=self.guidance_args.get('n_com', 2)).float().to(adj.device)
            self.Z = Zs.permute(0,2,1)
        loss_kwargs['Z'] = self.Z.clone()
        
        if self.guidance_args.method == 'greedy':
            n_traj = self.guidance_args.n_traj

            f, G = self.rsde.discretize(x, adj, flags, t, is_adj=is_adj)
            
            obj_mean = obj - f

            losses = torch.zeros(n_traj, obj.shape[0])
            obj_hats = []
            for i in range(n_traj):
                z = gen_noise(obj, flags, sym=is_adj)
                obj_hat = obj_mean.clone() + G[:, None, None] * z
                obj_hats.append(obj_hat)
                
                score_i = self.score_fn(x, obj_hat, flags, t+dt) if is_adj else self.score_fn(obj_hat, adj, flags, t+dt)

                obj0hat = self.sde.obj0estimation(obj_hat, score_i, timestep)
                obj0hat_masked = mask_adjs(obj0hat, flags) if is_adj else mask_x(obj0hat, flags)
                
                losses[i,:] = loss_fn(obj0hat_masked, **loss_kwargs)

            losses_expanded = torch.argmin(losses, dim=0).view(1, obj.shape[0], 1, 1).expand(1, obj.shape[0], obj.shape[1], obj.shape[2]).to(obj.device)

            return torch.gather(torch.stack(obj_hats, dim=0), 0, losses_expanded).squeeze(0), obj_mean

        elif self.guidance_args.method == 'zero':
            n_traj = self.guidance_args.n_traj

            f, G = self.rsde.discretize(x, adj, flags, t, is_adj=is_adj)
            
            obj_mean = obj - f

            score_no_noise = self.score_fn(x, obj_mean.clone(), flags, t) if is_adj else self.score_fn(obj.clone(), adj, flags, t)
            obj0hat_no_noise = self.sde.obj0estimation(obj_mean.clone(), score_no_noise, timestep)
            obj0hat_masked_no_noise = mask_adjs(obj0hat_no_noise, flags) if is_adj else mask_x(obj0hat_no_noise, flags)
            no_noise_loss = loss_fn(obj0hat_masked_no_noise, **loss_kwargs)

            losses = torch.zeros(n_traj, obj.shape[0], device=obj.device)
            noise_directions = []
            for i in range(n_traj):
                z = gen_noise(obj, flags, sym=is_adj)
                obj_hat = obj_mean.clone() + G[:, None, None] * z
                noise_directions.append(z)
                
                score_i = self.score_fn(x, obj_hat, flags, t+dt) if is_adj else self.score_fn(obj_hat, adj, flags, t+dt)

                obj0hat = self.sde.obj0estimation(obj_hat, score_i, timestep)
                obj0hat_masked = mask_adjs(obj0hat, flags) if is_adj else mask_x(obj0hat, flags)
                
                losses[i,:] = loss_fn(obj0hat_masked, **loss_kwargs)

            weights = (losses - no_noise_loss[None,:]) / G[None, :]
            directions = torch.stack(noise_directions, dim=0)
            obj = obj_mean - self.guidance_args.lr_zero * (weights[:,:,None,None] * directions).mean(dim=0)
            return obj, obj_mean

        
        elif self.guidance_args.method == 'loss':

            with torch.enable_grad():

                obj.requires_grad = True

                score = self.score_fn(x, obj, flags, t) if is_adj else self.score_fn(obj, adj, flags, t)

                obj0hat = self.sde.obj0estimation(obj, score, timestep)
                obj0hat_masked = mask_adjs(obj0hat, flags) if is_adj else mask_x(obj0hat, flags)

                loss = loss_fn(obj0hat_masked, **loss_kwargs).mean()

                loss.backward()

                obj_grad = obj.grad.detach().clone()
                obj.grad = None

            f, G = self.rsde.discretize(x, adj, flags, t, is_adj=is_adj)

            obj_mean = obj - f

            z = gen_noise(obj, flags, sym=is_adj)
            obj = obj_mean + G[:, None, None] * z
            
            if self.guidance_args.lr_guidance_method == 'adaptive':
                obj -= self.guidance_args.lr_guidance / torch.abs(loss) * obj_grad
            else:
                obj -= self.guidance_args.lr_guidance * obj_grad

            return obj, obj_mean
        else:
            raise NotImplementedError(f"guidance method {self.guidance_args.method} not yet supported.")


    def update_fn(self, x, adj, flags, t):
        timestep = (t[0] * (self.rsde.N - 1) / self.rsde.T).long()

        var = x if self.obj == 'x' else adj

        if self.guidance_args is not None and \
                timestep > 0:
            var, var_mean = self.guidance(x, adj, flags, t, is_adj=self.obj == 'adj')
        else:
            f, G = self.rsde.discretize(x, adj, flags, t, is_adj=self.obj == 'adj')
            z = gen_noise(var, flags, sym=self.obj == 'adj')
            var_mean = var - f
            var = var_mean + G[:, None, None] * z
        return var, var_mean


In [ ]:
# Repeated here for cleaner code
class EulerMaruyamaPredictor(Predictor):
    def __init__(self, obj, sde, score_fn, probability_flow=False, guidance_args=None):
        super().__init__(sde, score_fn, probability_flow)
        self.obj = obj

        self.guidance_args = guidance_args

        self.Z = None

    def guidance(self, x, adj, flags, t, is_adj):
        obj = adj if is_adj else x

        dt = -1. / self.rsde.N
        timestep = (t * (self.rsde.N - 1) / self.rsde.T).long()

        loss_fn = eval(self.guidance_args.loss_fn)
        loss_kwargs = self.guidance_args.get('loss_kwargs', {})

        if self.obj == 'adj' and self.Z is None:
            if self.guidance_args.method_Z == "communities":
                adj0 = self.sde.obj0estimation(adj, self.score_fn(x, adj, flags, t), timestep)
                adj0 = mask_adjs(adj0, flags)
                gs = adjs_to_graphs(quantize(adj0), True)
                Zs = torch.zeros((adj.shape[0], adj.shape[1], self.guidance_args.get('n_com', 2))).to(adj.device)
                for g, graph in enumerate(gs):
                    Zs[g,:graph.number_of_nodes(),:] = compute_group_assignments(graph, self.guidance_args.get('n_com', 2))
                self.Z = Zs.permute(0,2,1)
            elif self.guidance_args.method_Z == "random":
                idxs_com = np.random.randint(0, self.guidance_args.get('n_com', 2), size=(adj.shape[0], adj.shape[1]))
                Zs = torch.nn.functional.one_hot(torch.tensor(idxs_com), num_classes=self.guidance_args.get('n_com', 2)).float().to(adj.device)
                self.Z = Zs.permute(0,2,1)
        loss_kwargs['Z'] = self.Z.clone()
        
        if self.guidance_args.method == 'greedy':
            n_traj = self.guidance_args.n_traj

            drift, diffusion = self.rsde.sde(x, adj, flags, t, is_adj=is_adj)
            
            obj_mean = obj + drift * dt

            losses = torch.zeros(n_traj, obj.shape[0])
            obj_hats = []
            for i in range(n_traj):
                z = gen_noise(obj, flags, sym=is_adj)
                obj_hat = obj_mean.clone() + diffusion[:, None, None] * np.sqrt(-dt) * z
                obj_hats.append(obj_hat)
                
                score_i = self.score_fn(x, obj_hat, flags, t+dt) if is_adj else self.score_fn(obj_hat, adj, flags, t+dt)

                obj0hat = self.sde.obj0estimation(obj_hat, score_i, timestep)
                obj0hat_masked = mask_adjs(obj0hat, flags) if is_adj else mask_x(obj0hat, flags)
                
                losses[i,:] = loss_fn(obj0hat_masked, **loss_kwargs)

            losses_expanded = torch.argmin(losses, dim=0).view(1, obj.shape[0], 1, 1).expand(1, obj.shape[0], obj.shape[1], obj.shape[2]).to(obj.device)

            return torch.gather(torch.stack(obj_hats, dim=0), 0, losses_expanded).squeeze(0), obj_mean

        elif self.guidance_args.method == 'zero':
            n_traj = self.guidance_args.n_traj

            drift, diffusion = self.rsde.sde(x, adj, flags, t, is_adj=is_adj)
            
            obj_mean = obj + drift * dt

            z = gen_noise(obj, flags, sym=is_adj)
            obj = obj.clone() + diffusion[:, None, None] * np.sqrt(-dt) * z

            score_no_noise = self.score_fn(x, obj.clone(), flags, t+dt) if is_adj else self.score_fn(obj.clone(), adj, flags, t)
            obj0hat_no_noise = self.sde.obj0estimation(obj.clone(), score_no_noise, timestep)
            obj0hat_masked_no_noise = mask_adjs(obj0hat_no_noise, flags) if is_adj else mask_x(obj0hat_no_noise, flags)
            no_noise_loss = loss_fn(obj0hat_masked_no_noise, **loss_kwargs)

            losses = torch.zeros(n_traj, obj.shape[0], device=obj.device)
            noise_directions = []
            for i in range(n_traj):
                z = gen_noise(obj, flags, sym=is_adj)
                obj_hat = obj.clone() + self.guidance_args.delta * z
                noise_directions.append(z)
                
                score_i = self.score_fn(x, obj_hat, flags, t+dt) if is_adj else self.score_fn(obj_hat, adj, flags, t+dt)

                obj0hat = self.sde.obj0estimation(obj_hat, score_i, timestep)
                obj0hat_masked = mask_adjs(obj0hat, flags) if is_adj else mask_x(obj0hat, flags)
                
                losses[i,:] = loss_fn(obj0hat_masked, **loss_kwargs)

            weights = (losses - no_noise_loss[None,:]) / self.guidance_args.delta # (diffusion[None, :] * np.sqrt(-dt))
            directions = torch.stack(noise_directions, dim=0)
            weighted_directions = (weights[:,:,None,None] * directions).mean(dim=0)

            if self.guidance_args.clip_method == "clip":
                # Gradient clipping
                weighted_directions = torch.clamp(weighted_directions, -self.guidance_args.clip, self.guidance_args.clip)

                obj = obj - self.guidance_args.lr_zero * weighted_directions
            elif self.guidance_args.clip_method == "norm":
                norm_grad_step = torch.linalg.norm(weighted_directions, dim=(1,2)) / (weighted_directions.shape[1] * weighted_directions.shape[2])
                norm_grad_step = torch.where(norm_grad_step < 1e-7, torch.ones_like(norm_grad_step), norm_grad_step)

                obj = obj - self.guidance_args.lr_zero * weighted_directions / norm_grad_step[:,None,None]
            else:
                obj = obj - self.guidance_args.lr_zero * weighted_directions
            return obj, obj_mean
        
        elif self.guidance_args.method == 'loss':

            with torch.enable_grad():

                obj.requires_grad = True

                score = self.score_fn(x, obj, flags, t) if is_adj else self.score_fn(obj, adj, flags, t)

                obj0hat = self.sde.obj0estimation(obj, score, timestep)
                obj0hat_masked = mask_adjs(obj0hat, flags) if is_adj else mask_x(obj0hat, flags)

                loss = loss_fn(obj0hat_masked, **loss_kwargs).mean()

                loss.backward()

                obj_grad = obj.grad.detach().clone()
                obj.grad = None

            drift, diffusion = self.sde.sde(adj, t)
            drift = drift - diffusion[:, None, None] ** 2 * score * (0.5 if self.rsde.probability_flow else 1.)
            # -------- Set the diffusion function to zero for ODEs. --------
            diffusion = 0. if self.rsde.probability_flow else diffusion

            obj_mean = obj + drift * dt

            z = gen_noise(obj, flags, sym=is_adj)
            obj = obj_mean + diffusion[:, None, None] * np.sqrt(-dt) * z

            if self.guidance_args.lr_guidance_method == 'adaptive':
                obj -= self.guidance_args.lr_guidance / torch.abs(loss) * obj_grad
            else:
                obj -= self.guidance_args.lr_guidance * obj_grad

            return obj, obj_mean
        else:
            raise NotImplementedError(f"guidance method {self.guidance_args.method} not yet supported.")

            
    def update_fn(self, x, adj, flags, t):
        dt = -1. / self.rsde.N
        timestep = (t[0] * (self.rsde.N - 1) / self.rsde.T).long()

        var = x if self.obj == 'x' else adj

        cond_guidance = self.guidance_args is not None and timestep > 0
        if self.guidance_args is not None and 'from_t' in self.guidance_args:
            cond_guidance = cond_guidance and timestep.item() < self.guidance_args.from_t
            
        if cond_guidance:
            var, var_mean = self.guidance(x, adj, flags, t, is_adj=self.obj == 'adj')
        else:
            z = gen_noise(var, flags, sym=self.obj == 'adj')
            drift, diffusion = self.rsde.sde(x, adj, flags, t, is_adj=self.obj == 'adj')
            var_mean = var + drift * dt
            var = var_mean + diffusion[:, None, None] * np.sqrt(-dt) * z
        return var, var_mean


In [ ]:
def sample(predictor_x, corrector_x,
           predictor_adj, corrector_adj,
           init_x=None, init_adj=None, flags=None):
    with torch.no_grad():
        # -------- Initial sample --------
        if init_x is not None:
            x = init_x.clone()
        else:
            x = predictor_x.sde.prior_sampling(shape_x).to(device_id)
        if init_adj is not None:
            adj = init_adj.clone()
        else:
            adj = predictor_adj.sde.prior_sampling_sym(shape_adj).to(device_id)
        
        x = mask_x(x, flags)
        adj = mask_adjs(adj, flags)
        diff_steps = predictor_adj.sde.N
        timesteps = torch.linspace(predictor_adj.sde.T, eps, diff_steps, device=device_id)

        # -------- Reverse diffusion process --------
        for i in trange(0, (diff_steps), desc = '[Sampling]', position = 1, leave=False):
            t = timesteps[i]
            vec_t = torch.ones(shape_adj[0], device=t.device) * t

            _x = x
            x, x_mean = corrector_x.update_fn(x, adj, flags, vec_t)
            adj, adj_mean = corrector_adj.update_fn(_x, adj, flags, vec_t)
            if torch.any(torch.isnan(adj)):
                break

            _x = x
            x, x_mean = predictor_x.update_fn(x, adj, flags, vec_t)
            adj, adj_mean = predictor_adj.update_fn(_x, adj, flags, vec_t)
    samples_int = quantize_mol(adj)

    # adj = torch.nn.functional.one_hot(torch.tensor(samples_int), num_classes=4).permute(0, 3, 1, 2)
    x = torch.where(x > 0.5, 1, 0)
    x = torch.concat([x, 1 - x.sum(dim=-1, keepdim=True)], dim=-1)      # 32, 9, 4 -> 32, 9, 5
    return samples_int, x

In [ ]:
def compute_group_assignments_tensor(adj, n_com):
    gs = adjs_to_graphs(adj, True)
    Zs = torch.zeros((adj.shape[0], adj.shape[1], n_com)).to(adj.device)
    for g, graph in enumerate(gs):
        Zs[g,:graph.number_of_nodes(),:] = compute_group_assignments(graph, n_com)
    return Zs.permute(0,2,1)

## Greedy

In [ ]:
method_Z = "random"

In [ ]:
guidance_config = load_yaml_config(f'config_guidance/fairness/{method_Z}/greedy.yaml')
guidance_args = edict({'method': 'greedy', 'obj': guidance_config['obj'], **guidance_config[configt.data.data.lower()]})

In [ ]:
score_fn_x = get_score_fn(sde_x, model_x, train=False, continuous=continuous)
score_fn_adj = get_score_fn(sde_adj, model_adj, train=False, continuous=continuous)

predictor_fn = ReverseDiffusionPredictor if predictor=='Reverse' else EulerMaruyamaPredictor 
corrector_fn = LangevinCorrector if corrector=='Langevin' else NoneCorrector

predictor_obj_x = predictor_fn('x', sde_x, score_fn_x, probability_flow)
corrector_obj_x = corrector_fn('x', sde_x, score_fn_x, snr, scale_eps, n_steps)

predictor_obj_adj = predictor_fn('adj', sde_adj, score_fn_adj, probability_flow, guidance_args=guidance_args)
corrector_obj_adj = corrector_fn('adj', sde_adj, score_fn_adj, snr, scale_eps, n_steps)

adj_greedy, x_greedy = sample(predictor_obj_x, corrector_obj_x, predictor_obj_adj, corrector_obj_adj, init_x, init_adj, init_flags_iter)

In [ ]:
Zs_greedy = predictor_obj_adj.Z.clone().to(device_id)

adj_greedy = torch.tensor(adj_greedy, device=device_id)

dp1_greedy = compute_dp1(adj_greedy, Zs_greedy).mean().item()
dp2_greedy = compute_dp2(adj_greedy, Zs_greedy).mean().item()
dp2_greedy_std = compute_dp2(adj_greedy, Zs_greedy).std().item()
nodedp1_greedy = compute_nodedp1(adj_greedy, Zs_greedy).mean().item()
nodedp2_greedy = compute_nodedp2(adj_greedy, Zs_greedy).mean().item()
nodedp2_greedy_std = compute_nodedp2(adj_greedy, Zs_greedy).std().item()
across_comm_edges_greedy = count_across_community_edges(adj_greedy, Zs_greedy).mean().item()

dp1_greedy, dp2_greedy, nodedp1_greedy, nodedp2_greedy, across_comm_edges_greedy

## Loss

In [ ]:
guidance_config = load_yaml_config(f'config_guidance/fairness/{method_Z}/loss.yaml')
guidance_args = edict({'method': 'loss', 'obj': guidance_config['obj'], **guidance_config[configt.data.data.lower()]})

In [ ]:
score_fn_x = get_score_fn(sde_x, model_x, train=False, continuous=continuous)
score_fn_adj = get_score_fn(sde_adj, model_adj, train=False, continuous=continuous)

predictor_fn = ReverseDiffusionPredictor if predictor=='Reverse' else EulerMaruyamaPredictor 
corrector_fn = LangevinCorrector if corrector=='Langevin' else NoneCorrector

predictor_obj_x = predictor_fn('x', sde_x, score_fn_x, probability_flow)
corrector_obj_x = corrector_fn('x', sde_x, score_fn_x, snr, scale_eps, n_steps)

predictor_obj_adj = predictor_fn('adj', sde_adj, score_fn_adj, probability_flow, guidance_args=guidance_args)
corrector_obj_adj = corrector_fn('adj', sde_adj, score_fn_adj, snr, scale_eps, n_steps)

adj_loss, x_loss = sample(predictor_obj_x, corrector_obj_x, predictor_obj_adj, corrector_obj_adj, init_x, init_adj, init_flags_iter)

In [ ]:
Zs_loss = predictor_obj_adj.Z.clone().to(device_id)

adj_loss = torch.tensor(adj_loss, device=device_id)

dp1_loss = compute_dp1(adj_loss, Zs_loss).mean().item()
dp2_loss = compute_dp2(adj_loss, Zs_loss).mean().item()
dp2_loss_std = compute_dp2(adj_loss, Zs_loss).std().item()
nodedp1_loss = compute_nodedp1(adj_loss, Zs_loss).mean().item()
nodedp2_loss = compute_nodedp2(adj_loss, Zs_loss).mean().item()
nodedp2_loss_std = compute_nodedp2(adj_loss, Zs_loss).std().item()
across_comm_edges_loss = count_across_community_edges(adj_loss, Zs_loss).mean().item()

dp1_loss, dp2_loss, nodedp1_loss, nodedp2_loss, across_comm_edges_loss

## Zero order optimization

In [ ]:
guidance_config = load_yaml_config(f'config_guidance/fairness/{method_Z}/zero.yaml')
guidance_args = edict({'method': 'zero', 'obj': guidance_config['obj'], **guidance_config[configt.data.data.lower()]})

In [ ]:
score_fn_x = get_score_fn(sde_x, model_x, train=False, continuous=continuous)
score_fn_adj = get_score_fn(sde_adj, model_adj, train=False, continuous=continuous)

predictor_fn = ReverseDiffusionPredictor if predictor=='Reverse' else EulerMaruyamaPredictor 
corrector_fn = LangevinCorrector if corrector=='Langevin' else NoneCorrector

predictor_obj_x = predictor_fn('x', sde_x, score_fn_x, probability_flow)
corrector_obj_x = corrector_fn('x', sde_x, score_fn_x, snr, scale_eps, n_steps)

predictor_obj_adj = predictor_fn('adj', sde_adj, score_fn_adj, probability_flow, guidance_args=guidance_args)
corrector_obj_adj = corrector_fn('adj', sde_adj, score_fn_adj, snr, scale_eps, n_steps)

adj_zero, x_zero = sample(predictor_obj_x, corrector_obj_x, predictor_obj_adj, corrector_obj_adj, init_x, init_adj, init_flags_iter)

In [ ]:
Zs_zero = predictor_obj_adj.Z.clone().to(device_id)

adj_zero = torch.tensor(adj_zero, device=device_id)

dp1_zero = compute_dp1(adj_zero, Zs_zero).mean().item()
dp2_zero = compute_dp2(adj_zero, Zs_zero).mean().item()
dp2_zero_std = compute_dp2(adj_zero, Zs_zero).std().item()
nodedp1_zero = compute_nodedp1(adj_zero, Zs_zero).mean().item()
nodedp2_zero = compute_nodedp2(adj_zero, Zs_zero).mean().item()
nodedp2_zero_std = compute_nodedp2(adj_zero, Zs_zero).std().item()
across_comm_edges_zero = count_across_community_edges(adj_zero, Zs_zero).mean().item()

dp1_zero, dp2_zero, nodedp1_zero, nodedp2_zero, across_comm_edges_zero

## DiGress

In [ ]:
import sys
sys.path.append('DiGress')
sys.path.append('DiGress/src')

from guided_sampling import sample_fair_digress

In [ ]:
guidance_config_digress = load_yaml_config(f'config_guidance/fairness/{method_Z}/digress.yaml')
digress_lambda = guidance_config_digress[configt.data.data.lower()]['guidance_lambda']
digress_base_config_path = guidance_config_digress[configt.data.data.lower()]['base_config_path']
digress_ckpt_path = guidance_config_digress[configt.data.data.lower()]['ckpt_path']

In [ ]:
x_digress, adj_digress, Zs_digress = sample_fair_digress(batch_size, digress_base_config_path, digress_ckpt_path, device_id, guidance_lambda=digress_lambda)

In [ ]:
dp1_digress = compute_dp1(adj_digress.to(device_id), Zs_digress).mean().item()
dp2_digress = compute_dp2(adj_digress.to(device_id), Zs_digress).mean().item()
dp2_digress_std = compute_dp2(adj_digress.to(device_id), Zs_digress).std().item()
nodedp1_digress = compute_nodedp1(adj_digress.to(device_id), Zs_digress).mean().item()
nodedp2_digress = compute_nodedp2(adj_digress.to(device_id), Zs_digress).mean().item()
nodedp2_digress_std = compute_nodedp2(adj_digress.to(device_id), Zs_digress).std().item()
across_comm_edges_digress = count_across_community_edges(adj_digress.to(device_id), Zs_digress).mean().item()

dp1_digress, dp2_digress, nodedp1_digress, nodedp2_digress, across_comm_edges_digress

## DiGress without guidance

In [ ]:
digress_lambda = 0.
x_digress_unguided, adj_digress_unguided, Zs_digress_unguided = sample_fair_digress(batch_size, digress_base_config_path, digress_ckpt_path, device_id, guidance_lambda=digress_lambda)

In [ ]:
dp1_digress_unguided = compute_dp1(adj_digress_unguided.to(device_id), Zs_digress_unguided).mean().item()
dp2_digress_unguided = compute_dp2(adj_digress_unguided.to(device_id), Zs_digress_unguided).mean().item()
dp2_digress_unguided_std = compute_dp2(adj_digress_unguided.to(device_id), Zs_digress_unguided).std().item()
nodedp1_digress_unguided = compute_nodedp1(adj_digress_unguided.to(device_id), Zs_digress_unguided).mean().item()
nodedp2_digress_unguided = compute_nodedp2(adj_digress_unguided.to(device_id), Zs_digress_unguided).mean().item()
nodedp2_digress_unguided_std = compute_nodedp2(adj_digress_unguided.to(device_id), Zs_digress_unguided).std().item()
across_comm_edges_digress_unguided = count_across_community_edges(adj_digress_unguided.to(device_id), Zs_digress_unguided).mean().item()

dp1_digress_unguided, dp2_digress_unguided, nodedp1_digress_unguided, nodedp2_digress_unguided, across_comm_edges_digress_unguided

## Unconstrained

In [ ]:
score_fn_x = get_score_fn(sde_x, model_x, train=False, continuous=continuous)
score_fn_adj = get_score_fn(sde_adj, model_adj, train=False, continuous=continuous)

predictor_fn = ReverseDiffusionPredictor if predictor=='Reverse' else EulerMaruyamaPredictor 
corrector_fn = LangevinCorrector if corrector=='Langevin' else NoneCorrector

predictor_obj_x = predictor_fn('x', sde_x, score_fn_x, probability_flow)
corrector_obj_x = corrector_fn('x', sde_x, score_fn_x, snr, scale_eps, n_steps)

predictor_obj_adj = predictor_fn('adj', sde_adj, score_fn_adj, probability_flow)
corrector_obj_adj = corrector_fn('adj', sde_adj, score_fn_adj, snr, scale_eps, n_steps)

adj_uncons, x_uncons = sample(predictor_obj_x, corrector_obj_x, predictor_obj_adj, corrector_obj_adj, init_x, init_adj, init_flags_iter)

In [ ]:
# Assign half and half of the nodes randomly to each group
n_elems = adj_uncons.shape[1] // 2
# Create a template row with the correct number of elements
template_row = np.array([0] * n_elems + [1] * (adj_uncons.shape[1] - n_elems))

# Create an array where each row is a copy of the template row
idxs_com = np.tile(template_row, (adj_uncons.shape[0], 1))

# Apply a random permutation along the columns for each row
for i in range(adj_uncons.shape[0]):
    np.random.shuffle(idxs_com[i])
Zs = torch.nn.functional.one_hot(torch.tensor(idxs_com), num_classes=guidance_args.get('n_com', 2)).float().to(device_id)
Zs_uncons = Zs.permute(0,2,1)

adj_uncons = torch.tensor(adj_uncons, device=device_id)

dp1_uncons = compute_dp1(adj_uncons, Zs_uncons).mean().item()
dp2_uncons = compute_dp2(adj_uncons, Zs_uncons).mean().item()
dp2_uncons_std = compute_dp2(adj_uncons, Zs_uncons).std().item()
nodedp1_uncons = compute_nodedp1(adj_uncons, Zs_uncons).mean().item()
nodedp2_uncons = compute_nodedp2(adj_uncons, Zs_uncons).mean().item()
nodedp2_uncons_std = compute_nodedp2(adj_uncons, Zs_uncons).std().item()
across_comm_edges_uncons = count_across_community_edges(adj_uncons, Zs_uncons).mean().item()

dp1_uncons, dp2_uncons, nodedp1_uncons, nodedp2_uncons, across_comm_edges_uncons

# Evaluation

In [ ]:
import pandas as pd

In [ ]:
df_results = pd.DataFrame({
    'Method': ['GGDiff-G', 'GGDiff-C', 'GGDiff-Z', 'DiGress', 'Uncons. (DiGress)', 'Uncons. (GGDS)'],
    'DP1': [dp1_loss, dp1_greedy, dp1_zero, dp1_digress, dp1_digress_unguided, dp1_uncons],
    'DP2': [dp2_loss, dp2_greedy, dp2_zero, dp2_digress, dp2_digress_unguided, dp2_uncons],
    'Node DP1': [nodedp1_loss, nodedp1_greedy, nodedp1_zero, nodedp1_digress, nodedp1_digress_unguided, nodedp1_uncons],
    'Node DP2': [nodedp2_loss, nodedp2_greedy, nodedp2_zero, nodedp2_digress, nodedp2_digress_unguided, nodedp2_uncons]
}).set_index('Method')
# format the float numbers to 4 decimal places
df_results.style.format({
    'DP1': "{:.4f}",
    'DP2': "{:.4f}",
    'Node DP1': "{:.4f}",
    'Node DP2': "{:.4f}"
})

In [ ]:
min_vals = df_results.min(axis=0)

print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Results for the fair graph generation experiment.}")
print("\\label{tab:fair_graph_gen}")
print("\\begin{tabular}{cccc}")
print("\\toprule")
print("\\textbf{Method} & \\textbf{$\\Delta$ DP} & \\textbf{$\\Delta \\text{DP}_{\\text{node}}$ } \\\\")
print("\\midrule")
for index, row in df_results.iterrows():
    print(f"{index} & ", end="")
    for metric in ['DP2', 'Node DP2']:
        val = row[metric]
        if val == min_vals[metric]:
            print(f"\\textbf{{{val:.4f}}} & ", end="")
        else:
            print(f"{val:.4f} & ", end="")
    print("\\\\")
print("\\bottomrule")
print("\\end{tabular}")
print("\\end{table}")